# Analyse politique finale — Sentiment envers le président (1973-1993)

Visualisations publication-ready à partir de `leaders_sentiment_corpus.xlsx` (6 342 mentions classifiées).

**Figures produites :**
1. Vue d'ensemble (2×2) : président, temporel, bloc, heatmap
2. Évolution temporelle par bloc politique
3. Top partis critiques envers Mitterrand
4. Montée en puissance du Front National
5. Analyse géographique par département

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.gridspec import GridSpec

NOTEBOOK_DIR = Path(os.getcwd())
ROOT_DIR     = NOTEBOOK_DIR.parent
OUTPUT_DIR   = ROOT_DIR / 'data' / 'results' / 'output_best_model'
GRAPHS_DIR   = OUTPUT_DIR / 'graphs'
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

# ── Style global ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
    'axes.labelsize':   11,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'legend.frameon':   False,
    'figure.dpi':       150,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
})

# ── Palettes ──────────────────────────────────────────────────────────────────
C_NEG   = '#e74c3c'
C_POS   = '#2ecc71'
C_BLOC  = {'gauche': '#c0392b', 'centre': '#f39c12', 'droite': '#2980b9', 'non classé': '#95a5a6'}
C_PRES  = {'Georges Pompidou': '#8e44ad', 'Valéry Giscard d\'Estaing': '#16a085', 'François Mitterrand': '#e67e22'}

LABEL_LIST = ['négatif', 'positif']
COLORS     = {'négatif': C_NEG, 'positif': C_POS}

ANNEE_LABELS = {1973: 'Pompidou\n1973', 1978: 'Giscard\n1978',
                1981: 'Mitterrand\n1981', 1988: 'Mitterrand\n1988', 1993: 'Mitterrand\n1993'}

print('Imports OK')

In [ ]:
# ── Chargement des données ────────────────────────────────────────────────────
df = pd.read_excel(OUTPUT_DIR / 'leaders_sentiment_corpus.xlsx')
print(f'Shape : {df.shape}')
print(f'Colonnes : {df.columns.tolist()}')
print()
print('Sentiment :'); print(df['sentiment_president'].value_counts())
print()
print('Présidents :'); print(df['actual_president'].value_counts())
print()
print('Blocs :'); print(df['bloc_politique'].value_counts())

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def pct_neg(sub):
    """% mentions négatives dans un sous-ensemble."""
    if len(sub) == 0:
        return np.nan
    return (sub['sentiment_president'] == 'négatif').sum() / len(sub) * 100

PRES_ORDER = ['Georges Pompidou', 'Valéry Giscard d\'Estaing', 'François Mitterrand']
PRES_SHORT = {'Georges Pompidou': 'Pompidou\n1973', 
               'Valéry Giscard d\'Estaing': "Giscard\n1978",
               'François Mitterrand': 'Mitterrand\n1981-1993'}
BLOC_ORDER  = ['gauche', 'centre', 'droite', 'non classé']
ANNEE_ORDER = [1973, 1978, 1981, 1988, 1993]

## Figure 1 — Vue d'ensemble (2×2)

In [ ]:
fig = plt.figure(figsize=(16, 11))
gs  = GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# ── 1a : % négatif par président ──────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
pres_pct = (
    df.groupby(['actual_president', 'sentiment_president'])
      .size().unstack(fill_value=0)
      .reindex(PRES_ORDER, fill_value=0)
)
pres_pct_norm = pres_pct.div(pres_pct.sum(axis=1), axis=0) * 100

x  = np.arange(len(PRES_ORDER))
w  = 0.55
b1 = ax1.bar(x, pres_pct_norm['négatif'],  width=w, color=C_NEG,  label='Négatif', zorder=3)
b2 = ax1.bar(x, pres_pct_norm['positif'],  width=w, color=C_POS,  label='Positif',
             bottom=pres_pct_norm['négatif'], zorder=3)

# Annotation % négatif
for rect, val in zip(b1, pres_pct_norm['négatif']):
    ax1.text(rect.get_x() + rect.get_width()/2, val/2,
             f'{val:.0f}%', ha='center', va='center', color='white', fontsize=10, fontweight='bold')

labels_short = [PRES_SHORT[p] for p in PRES_ORDER]
ax1.set_xticks(x); ax1.set_xticklabels(labels_short, fontsize=9)
ax1.set_ylabel('% mentions'); ax1.set_ylim(0, 105)
ax1.set_title('(a) Sentiment par président')
ax1.yaxis.set_major_formatter(mticker.PercentFormatter())
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(axis='y', alpha=0.3, zorder=0)

# ── 1b : Évolution % négatif par année ────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ann = (
    df.groupby(['annee', 'sentiment_president'])
      .size().unstack(fill_value=0)
      .reindex(ANNEE_ORDER, fill_value=0)
)
ann['pct_neg'] = ann['négatif'] / (ann['négatif'] + ann['positif']) * 100
ann['total']   = ann['négatif'] + ann['positif']

ax2.plot(range(len(ANNEE_ORDER)), ann['pct_neg'], 'o-', color='#2c3e50',
         linewidth=2.5, markersize=9, zorder=3)
for i, (yr, row) in enumerate(ann.iterrows()):
    ax2.annotate(f"{row['pct_neg']:.0f}%\n(n={row['total']:.0f})",
                 xy=(i, row['pct_neg']), xytext=(0, 14),
                 textcoords='offset points', ha='center', fontsize=8.5, color='#2c3e50')

# Zones présidents
zone_colors = ['#d7bde2', '#a9dfbf', '#fad7a0']
zones = [([0, 0], 'Pompidou'), ([1, 1], 'Giscard'), ([2, 4], 'Mitterrand')]
for (rng, label), color in zip(zones, zone_colors):
    ax2.axvspan(rng[0] - 0.4, rng[-1] + 0.4, alpha=0.25, color=color, zorder=0)
    ax2.text((rng[0] + rng[-1])/2, 5, label, ha='center', va='bottom', fontsize=8,
             color='#5d6d7e', style='italic')

ax2.set_xticks(range(len(ANNEE_ORDER)))
ax2.set_xticklabels([str(y) for y in ANNEE_ORDER])
ax2.set_ylabel('% mentions négatives')
ax2.set_ylim(0, 95); ax2.set_xlim(-0.5, 4.5)
ax2.set_title('(b) Évolution temporelle du sentiment négatif')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
ax2.grid(axis='y', alpha=0.3)

# ── 1c : % négatif par bloc ───────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
bloc = (
    df.groupby(['bloc_politique', 'sentiment_president'])
      .size().unstack(fill_value=0)
      .reindex(BLOC_ORDER, fill_value=0)
)
bloc_norm = bloc.div(bloc.sum(axis=1), axis=0) * 100
bloc['total'] = bloc.sum(axis=1)

x3 = np.arange(len(BLOC_ORDER))
b3 = ax3.bar(x3, bloc_norm['négatif'], width=0.55,
             color=[C_BLOC[b] for b in BLOC_ORDER], zorder=3, alpha=0.85)
for rect, bloc_name, val in zip(b3, BLOC_ORDER, bloc_norm['négatif']):
    n = bloc.loc[bloc_name, 'total']
    ax3.text(rect.get_x() + rect.get_width()/2, val + 1.5,
             f'{val:.0f}%\n(n={n:.0f})', ha='center', va='bottom', fontsize=8.5)

labels_bloc = ['Gauche', 'Centre', 'Droite', 'Non classé']
ax3.set_xticks(x3); ax3.set_xticklabels(labels_bloc)
ax3.set_ylabel('% mentions négatives'); ax3.set_ylim(0, 75)
ax3.set_title('(c) Sentiment négatif par bloc politique')
ax3.yaxis.set_major_formatter(mticker.PercentFormatter())
ax3.grid(axis='y', alpha=0.3, zorder=0)

# ── 1d : Heatmap président × bloc ─────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
heat = (
    df.groupby(['actual_president', 'bloc_politique', 'sentiment_president'])
      .size().unstack(fill_value=0)
)
for c in LABEL_LIST:
    if c not in heat.columns:
        heat[c] = 0
heat = heat.reindex(columns=LABEL_LIST, fill_value=0)
heat['pct_neg'] = heat['négatif'] / (heat['négatif'] + heat['positif']) * 100
heat_pivot = heat['pct_neg'].unstack('bloc_politique').reindex(
    index=PRES_ORDER, columns=BLOC_ORDER
)

# Annotations avec n
heat_n = (heat['négatif'] + heat['positif']).unstack('bloc_politique').reindex(
    index=PRES_ORDER, columns=BLOC_ORDER
)
annot = heat_pivot.round(0).astype(str) + '%\n(n=' + heat_n.fillna(0).astype(int).astype(str) + ')'

sns.heatmap(
    heat_pivot, annot=annot, fmt='', cmap='RdYlGn_r',
    vmin=0, vmax=100, ax=ax4,
    linewidths=0.8, linecolor='white',
    cbar_kws={'label': '% négatif', 'shrink': 0.8},
    annot_kws={'size': 8.5}
)
pres_labels = ['Pompidou\n1973', 'Giscard\n1978', 'Mitterrand\n1981-93']
bloc_labels  = ['Gauche', 'Centre', 'Droite', 'Non classé']
ax4.set_yticklabels(pres_labels, rotation=0, fontsize=9)
ax4.set_xticklabels(bloc_labels, rotation=15, fontsize=9)
ax4.set_xlabel(''); ax4.set_ylabel('')
ax4.set_title('(d) % mentions négatives : président × bloc')

fig.suptitle('Sentiment des candidats envers le président en exercice (1973-1993)',
             fontsize=15, fontweight='bold', y=1.01)

out1 = GRAPHS_DIR / 'fig1_vue_ensemble.png'
fig.savefig(out1)
plt.show()
print(f'Sauvegardé : {out1}')

## Figure 2 — Évolution temporelle par bloc politique

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── 2a : % négatif par (année × bloc) ─────────────────────────────────────────
ax = axes[0]

blocs_to_plot = ['gauche', 'centre', 'droite']
bloc_labels_map = {'gauche': 'Gauche', 'centre': 'Centre', 'droite': 'Droite'}

for bloc in blocs_to_plot:
    sub = df[df['bloc_politique'] == bloc]
    pts = []
    for yr in ANNEE_ORDER:
        s = sub[sub['annee'] == yr]
        pts.append(pct_neg(s) if len(s) >= 5 else np.nan)
    
    color = C_BLOC[bloc]
    ax.plot(range(len(ANNEE_ORDER)), pts, 'o-', color=color, linewidth=2.2,
            markersize=8, label=bloc_labels_map[bloc], zorder=3)
    for i, v in enumerate(pts):
        if not np.isnan(v):
            ax.annotate(f'{v:.0f}%', xy=(i, v), xytext=(0, 8),
                        textcoords='offset points', ha='center', fontsize=8, color=color)

# Ligne corpus complet
ann_total = [pct_neg(df[df['annee'] == yr]) for yr in ANNEE_ORDER]
ax.plot(range(len(ANNEE_ORDER)), ann_total, '--', color='#2c3e50', linewidth=1.5,
        alpha=0.5, label='Corpus complet', zorder=2)

# Zones présidents
bg_colors = ['#e8daef', '#d5f5e3', '#fdebd0']
bg_zones  = [([0, 0], 'Pompidou'), ([1, 1], 'Giscard'), ([2, 4], 'Mitterrand')]
for (rng, lbl), color in zip(bg_zones, bg_colors):
    ax.axvspan(rng[0] - 0.45, rng[-1] + 0.45, alpha=0.3, color=color, zorder=0)
    ax.text((rng[0] + rng[-1])/2, 2, lbl, ha='center', va='bottom',
            fontsize=9, color='#5d6d7e', style='italic')

ax.set_xticks(range(len(ANNEE_ORDER)))
ax.set_xticklabels([str(y) for y in ANNEE_ORDER])
ax.set_ylabel('% mentions négatives')
ax.set_ylim(-5, 110); ax.set_xlim(-0.5, 4.5)
ax.set_title('(a) % négatif par année et par bloc politique')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.25)

# ── 2b : Heatmap année × bloc ─────────────────────────────────────────────────
ax2 = axes[1]

heatmap_data = []
for yr in ANNEE_ORDER:
    row = {}
    for bloc in BLOC_ORDER:
        s = df[(df['annee'] == yr) & (df['bloc_politique'] == bloc)]
        row[bloc] = pct_neg(s) if len(s) >= 5 else np.nan
    heatmap_data.append(row)

hm_df = pd.DataFrame(heatmap_data, index=ANNEE_ORDER, columns=BLOC_ORDER)

# Compter les n
hm_n = pd.DataFrame([
    {bloc: len(df[(df['annee']==yr) & (df['bloc_politique']==bloc)]) for bloc in BLOC_ORDER}
    for yr in ANNEE_ORDER
], index=ANNEE_ORDER, columns=BLOC_ORDER)

annot2 = hm_df.round(0).fillna(0).astype(int).astype(str) + '%\n(n=' + hm_n.astype(str) + ')'
annot2 = annot2.where(hm_n >= 5, other='—')

sns.heatmap(
    hm_df, annot=annot2, fmt='', cmap='RdYlGn_r',
    vmin=0, vmax=100, ax=ax2,
    linewidths=0.8, linecolor='white',
    cbar_kws={'label': '% négatif', 'shrink': 0.8},
    annot_kws={'size': 8.5}
)

# Étiquettes axes
yr_labels = ['1973\n(Pompidou)', '1978\n(Giscard)', '1981\n(Mitterrand)',
             '1988\n(Mitterrand)', '1993\n(Mitterrand)']
ax2.set_yticklabels(yr_labels, rotation=0, fontsize=9)
ax2.set_xticklabels(['Gauche', 'Centre', 'Droite', 'Non classé'], rotation=15, fontsize=9)
ax2.set_xlabel(''); ax2.set_ylabel('')
ax2.set_title('(b) Heatmap % négatif : année × bloc politique')

fig.suptitle('Évolution du sentiment négatif par bloc politique (1973-1993)',
             fontsize=14, fontweight='bold')

out2 = GRAPHS_DIR / 'fig2_evolution_bloc.png'
fig.savefig(out2)
plt.show()
print(f'Sauvegardé : {out2}')

## Figure 3 — Top partis critiques envers Mitterrand

In [ ]:
# ── Nettoyage des partis (premier parti si composite) ─────────────────────────
def clean_parti(s):
    if not isinstance(s, str):
        return 'Non mentionné'
    first = s.split(';')[0].strip()
    return first if first else 'Non mentionné'

df['parti_clean'] = df['parti'].apply(clean_parti)

mitt_neg = df[(df['actual_president'] == 'François Mitterrand') &
              (df['sentiment_president'] == 'négatif')].copy()
mitt_pos = df[(df['actual_president'] == 'François Mitterrand') &
              (df['sentiment_president'] == 'positif')].copy()

# Top partis négatifs (exclure 'non mentionné' et 'inconnu')
EXCLUDE = {'non mentionné', 'inconnu', 'Non mentionné', 'Inconnu'}

top_neg = (
    mitt_neg[~mitt_neg['parti_clean'].isin(EXCLUDE)]
    ['parti_clean'].value_counts().head(15)
)
top_pos = (
    mitt_pos[~mitt_pos['parti_clean'].isin(EXCLUDE)]
    ['parti_clean'].value_counts().head(15)
)

print('Top négatifs Mitterrand:')
print(top_neg.to_string())
print('\nTop positifs Mitterrand:')
print(top_pos.to_string())

In [ ]:
# ── Calcul % négatif par parti (avec min 10 mentions) ──────────────────────────
mitt = df[df['actual_president'] == 'François Mitterrand'].copy()
mitt_p = mitt[~mitt['parti_clean'].isin(EXCLUDE)]

parti_stats = (
    mitt_p.groupby('parti_clean')['sentiment_president']
    .value_counts().unstack(fill_value=0)
)
for c in ['négatif', 'positif']:
    if c not in parti_stats.columns:
        parti_stats[c] = 0

parti_stats['total']   = parti_stats['négatif'] + parti_stats['positif']
parti_stats['pct_neg'] = parti_stats['négatif'] / parti_stats['total'] * 100
parti_stats = parti_stats[parti_stats['total'] >= 10].sort_values('négatif', ascending=False)

top15_neg = parti_stats.head(15)
print(top15_neg[['négatif', 'positif', 'total', 'pct_neg']].to_string())

In [ ]:
# ── Colorier par bloc ─────────────────────────────────────────────────────────
GAUCHE_KW = ['socialiste', 'communiste', 'lutte ouvrière', 'psu', 'radicaux de gauche',
             'sfio', 'gauche', 'trotsk', 'ligue communiste', 'travailleurs']
DROITE_KW = ['rassemblement pour la', 'rpr', 'républicain indépendant',
             'union des républicains', 'front national', 'alliance républicaine',
             'gaulliste', 'cnip', 'droite']
CENTRE_KW = ['union pour la démocratie', 'udf', 'centre démocrate', 'mrp', 'cds',
             'centre', 'réformateur', 'libéral', 'radical']

def bloc_from_parti(parti):
    if not isinstance(parti, str):
        return 'non classé'
    p = parti.lower()
    if any(k in p for k in GAUCHE_KW): return 'gauche'
    if any(k in p for k in DROITE_KW): return 'droite'
    if any(k in p for k in CENTRE_KW): return 'centre'
    return 'non classé'

top15_neg = top15_neg.copy()
top15_neg['bloc'] = [bloc_from_parti(p) for p in top15_neg.index]
bar_colors = [C_BLOC[b] for b in top15_neg['bloc']]

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={'width_ratios': [1.4, 1]})

# 3a : Nombre absolu de mentions négatives
ax = axes[0]
y_pos = range(len(top15_neg) - 1, -1, -1)
bars = ax.barh(list(y_pos), top15_neg['négatif'].values, color=bar_colors, alpha=0.85, height=0.7)

for bar, (_, row) in zip(bars, top15_neg.iterrows()):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f"{row['pct_neg']:.0f}% neg", va='center', fontsize=8.5, color='#555')

# Noms courts
short_names = {
    'Parti communiste français': 'PCF',
    'Lutte ouvrière': 'LO',
    'Front national': 'FN',
    'Parti socialiste': 'PS',
    'Rassemblement pour la République': 'RPR',
    'Parti socialiste unifié': 'PSU',
    'Parti des travailleurs': 'PT',
    'Union pour la démocratie française': 'UDF',
    'Mouvement des radicaux de gauche': 'MRG',
}
labels = [short_names.get(p, p[:30]) for p in top15_neg.index]
ax.set_yticks(list(y_pos)); ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Nombre de mentions négatives envers Mitterrand')
ax.set_title('(a) Top 15 partis critiques envers Mitterrand\n(mentions négatives absolues)')
ax.grid(axis='x', alpha=0.3)

# Légende blocs
patches = [mpatches.Patch(color=C_BLOC[b], label=b.capitalize()) for b in C_BLOC]
ax.legend(handles=patches, title='Bloc politique', loc='lower right', fontsize=8.5)

# 3b : % négatif (top 15 partis avec ≥10 mentions)
ax2 = axes[1]
pct_sorted = parti_stats[parti_stats['total'] >= 10].sort_values('pct_neg', ascending=False).head(15)
pct_sorted['bloc'] = [bloc_from_parti(p) for p in pct_sorted.index]
bar_colors2 = [C_BLOC[b] for b in pct_sorted['bloc']]
y2 = range(len(pct_sorted) - 1, -1, -1)

ax2.barh(list(y2), pct_sorted['pct_neg'].values, color=bar_colors2, alpha=0.85, height=0.7)
for i, (_, row) in enumerate(pct_sorted.iterrows()):
    ax2.text(row['pct_neg'] + 0.5, len(pct_sorted) - 1 - i,
             f"n={row['total']:.0f}", va='center', fontsize=8, color='#555')

labels2 = [short_names.get(p, p[:30]) for p in pct_sorted.index]
ax2.set_yticks(list(y2)); ax2.set_yticklabels(labels2, fontsize=9)
ax2.set_xlabel('% mentions négatives')
ax2.set_xlim(0, 115)
ax2.set_title('(b) % négatif par parti\n(partis avec ≥10 mentions)')
ax2.xaxis.set_major_formatter(mticker.PercentFormatter())
ax2.axvline(100, color='#e74c3c', linestyle='--', alpha=0.4, linewidth=1)
ax2.grid(axis='x', alpha=0.3)

fig.suptitle('Partis politiques critiques envers François Mitterrand (1981-1993)',
             fontsize=14, fontweight='bold')
plt.tight_layout()

out3 = GRAPHS_DIR / 'fig3_top_partis_mitterrand.png'
fig.savefig(out3)
plt.show()
print(f'Sauvegardé : {out3}')

## Figure 4 — Montée en puissance du Front National

In [ ]:
fn_df = df[df['parti_clean'].str.lower() == 'front national'].copy()
print('FN total:', len(fn_df))
print(fn_df.groupby(['annee','sentiment_president']).size().unstack(fill_value=0))

In [ ]:
fn_by_year = (
    fn_df.groupby(['annee','sentiment_president'])
    .size().unstack(fill_value=0)
    .reindex(ANNEE_ORDER, fill_value=0)
)
for c in ['négatif', 'positif']:
    if c not in fn_by_year.columns:
        fn_by_year[c] = 0
fn_by_year['total']   = fn_by_year['négatif'] + fn_by_year['positif']
fn_by_year['pct_neg'] = (fn_by_year['négatif'] / fn_by_year['total'].replace(0, np.nan) * 100)

# Corpus global par année pour comparaison
corpus_by_year = (
    df.groupby(['annee','sentiment_president'])
    .size().unstack(fill_value=0)
    .reindex(ANNEE_ORDER, fill_value=0)
)
corpus_by_year['total'] = corpus_by_year['négatif'] + corpus_by_year['positif']
corpus_by_year['pct_neg'] = corpus_by_year['négatif'] / corpus_by_year['total'] * 100
corpus_by_year['pct_fn'] = fn_by_year['total'] / corpus_by_year['total'] * 100

print(fn_by_year[['négatif','positif','total','pct_neg']])
print('\n% FN dans corpus global :')
print(corpus_by_year['pct_fn'])

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 6))

x = np.arange(len(ANNEE_ORDER))
w = 0.4

# Barres empilées FN (négatif + positif)
b_neg = ax1.bar(x, fn_by_year['négatif'], width=w, color=C_NEG, alpha=0.85, label='Mentions négatives FN', zorder=3)
b_pos = ax1.bar(x, fn_by_year['positif'], width=w, color=C_POS, alpha=0.85,
                bottom=fn_by_year['négatif'], label='Mentions positives FN', zorder=3)

# Annotations total
for i, (yr, row) in enumerate(fn_by_year.iterrows()):
    if row['total'] > 0:
        ax1.text(i, row['total'] + 6, f"n={row['total']:.0f}",
                 ha='center', fontsize=9, color='#2c3e50')

ax1.set_ylabel('Nombre de mentions (FN)', color='#2c3e50')
ax1.set_xticks(x)
ax1.set_xticklabels([str(y) for y in ANNEE_ORDER])
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(axis='y', alpha=0.25, zorder=0)
ax1.set_ylim(0, 700)

# Axe secondaire : % FN dans le corpus
ax2 = ax1.twinx()
ax2.spines['right'].set_visible(True)
pct_fn_vals = corpus_by_year['pct_fn'].values
ax2.plot(x, pct_fn_vals, 's--', color='#8e44ad', linewidth=2, markersize=8,
         label='% FN dans corpus', zorder=4)
for i, v in enumerate(pct_fn_vals):
    ax2.annotate(f'{v:.1f}%', xy=(i, v), xytext=(6, 3),
                 textcoords='offset points', fontsize=8.5, color='#8e44ad')
ax2.set_ylabel('% FN dans le corpus annuel', color='#8e44ad')
ax2.tick_params(axis='y', labelcolor='#8e44ad')
ax2.set_ylim(0, 20)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
ax2.legend(loc='upper center', fontsize=9)

# Zones présidents
bg_colors = ['#e8daef', '#d5f5e3', '#fdebd0']
bg_zones  = [([0, 0], 'Pompidou'), ([1, 1], 'Giscard'), ([2, 4], 'Mitterrand')]
for (rng, lbl), color in zip(bg_zones, bg_colors):
    ax1.axvspan(rng[0] - 0.45, rng[-1] + 0.45, alpha=0.2, color=color, zorder=0)
    ax1.text((rng[0] + rng[-1])/2, 640, lbl, ha='center', va='top',
             fontsize=9, color='#5d6d7e', style='italic')

ax1.set_title('Montée en puissance du Front National dans les professions de foi (1973-1993)',
              fontsize=13, fontweight='bold', pad=12)
ax1.set_xlabel('Année électorale')

out4 = GRAPHS_DIR / 'fig4_montee_fn.png'
fig.savefig(out4)
plt.show()
print(f'Sauvegardé : {out4}')

## Figure 5 — Analyse géographique par département

In [ ]:
# ── Stats par département (min 20 mentions) ───────────────────────────────────
geo = df.groupby(['departement', 'sentiment_president']).size().unstack(fill_value=0)
for c in ['négatif', 'positif']:
    if c not in geo.columns: geo[c] = 0
geo['total']   = geo['négatif'] + geo['positif']
geo['pct_neg'] = geo['négatif'] / geo['total'] * 100
geo = geo[geo['total'] >= 20].sort_values('pct_neg', ascending=False)
geo = geo[geo.index != 'inconnu']  # exclure 'inconnu'

print(f'Départements retenus (≥20 mentions) : {len(geo)}')
print('\nTop 10 :')
print(geo.head(10)[['négatif','total','pct_neg']].to_string())
print('\nBottom 10 :')
print(geo.tail(10)[['négatif','total','pct_neg']].to_string())

In [ ]:
# Top 20 + Bottom 15 pour la figure
geo_top = geo.head(20)
geo_bot = geo.tail(15)
geo_sel = pd.concat([geo_top, geo_bot])

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# ── 5a : Top 20 (les plus critiques) ──────────────────────────────────────────
ax = axes[0]
vals = geo_top['pct_neg'].values[::-1]
names = [str(n) for n in geo_top.index[::-1]]
totals = geo_top['total'].values[::-1]

cmap = plt.cm.RdYlGn_r
norm = plt.Normalize(vmin=0, vmax=100)
colors_geo = [cmap(norm(v)) for v in vals]

y_pos = range(len(vals))
bars = ax.barh(list(y_pos), vals, color=colors_geo, alpha=0.9, height=0.75)
for bar, v, n in zip(bars, vals, totals):
    ax.text(v + 0.5, bar.get_y() + bar.get_height()/2,
            f'{v:.1f}% (n={n:.0f})', va='center', fontsize=8)

ax.set_yticks(list(y_pos)); ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel('% mentions négatives envers le président')
ax.set_xlim(0, 95)
ax.set_title('(a) Top 20 départements\nles plus critiques', fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.axvline(geo['pct_neg'].mean(), color='#2c3e50', linestyle='--', alpha=0.5, linewidth=1.5)
ax.text(geo['pct_neg'].mean() + 0.5, 0.3, f'Moy. {geo["pct_neg"].mean():.0f}%',
        fontsize=8, color='#2c3e50', alpha=0.7)
ax.grid(axis='x', alpha=0.25)

# ── 5b : Bottom 15 (les moins critiques) ──────────────────────────────────────
ax2 = axes[1]
vals2  = geo_bot['pct_neg'].values[::-1]
names2 = [str(n) for n in geo_bot.index[::-1]]
tots2  = geo_bot['total'].values[::-1]

colors_geo2 = [cmap(norm(v)) for v in vals2]
y_pos2 = range(len(vals2))
bars2 = ax2.barh(list(y_pos2), vals2, color=colors_geo2, alpha=0.9, height=0.75)
for bar, v, n in zip(bars2, vals2, tots2):
    ax2.text(v + 0.5, bar.get_y() + bar.get_height()/2,
             f'{v:.1f}% (n={n:.0f})', va='center', fontsize=8)

ax2.set_yticks(list(y_pos2)); ax2.set_yticklabels(names2, fontsize=9)
ax2.set_xlabel('% mentions négatives envers le président')
ax2.set_xlim(0, 95)
ax2.set_title('(b) Bottom 15 départements\nles moins critiques', fontsize=12, fontweight='bold')
ax2.xaxis.set_major_formatter(mticker.PercentFormatter())
ax2.axvline(geo['pct_neg'].mean(), color='#2c3e50', linestyle='--', alpha=0.5, linewidth=1.5)
ax2.grid(axis='x', alpha=0.25)

fig.suptitle('Variation géographique du sentiment négatif envers le président (corpus complet)',
             fontsize=13, fontweight='bold')
plt.tight_layout()

out5 = GRAPHS_DIR / 'fig5_geographie.png'
fig.savefig(out5)
plt.show()
print(f'Sauvegardé : {out5}')

## Figure 6 — Analyse géographique avec contrôle Mitterrand uniquement

In [ ]:
# Contrôle : même analyse mais uniquement pour Mitterrand (éliminer l'effet composition)
df_mitt = df[df['actual_president'] == 'François Mitterrand'].copy()

geo_m = df_mitt.groupby(['departement','sentiment_president']).size().unstack(fill_value=0)
for c in ['négatif','positif']:
    if c not in geo_m.columns: geo_m[c] = 0
geo_m['total']   = geo_m['négatif'] + geo_m['positif']
geo_m['pct_neg'] = geo_m['négatif'] / geo_m['total'] * 100
geo_m = geo_m[geo_m['total'] >= 15]
geo_m = geo_m[geo_m.index != 'inconnu'].sort_values('pct_neg', ascending=False)

print(f'Départements retenus (Mitterrand, ≥15 mentions) : {len(geo_m)}')
print(geo_m.head(20)[['négatif','total','pct_neg']].to_string())

In [ ]:
# Figure comparative : sentiment Mitterrand par bloc × département (top 25)
# On croise dept × bloc pour voir si la variation géo est portée par un bloc particulier

dept_bloc = (
    df_mitt.groupby(['departement','bloc_politique','sentiment_president'])
    .size().unstack(fill_value=0)
)
for c in ['négatif','positif']:
    if c not in dept_bloc.columns: dept_bloc[c] = 0
dept_bloc['total']   = dept_bloc['négatif'] + dept_bloc['positif']
dept_bloc['pct_neg'] = dept_bloc['négatif'] / dept_bloc['total'] * 100
dept_bloc = dept_bloc[dept_bloc['total'] >= 5]

# Pivot pour heatmap : top 20 depts × blocs
top20_depts = geo_m.head(25).index.tolist()
hm_db = dept_bloc['pct_neg'].unstack('bloc_politique').reindex(
    index=top20_depts, columns=['gauche','centre','droite']
)

fig, axes = plt.subplots(1, 2, figsize=(16, 9))

# 6a : Heatmap dept × bloc (top 25 depts les plus critiques de Mitterrand)
ax = axes[0]
hm_n_db = dept_bloc['total'].unstack('bloc_politique').reindex(
    index=top20_depts, columns=['gauche','centre','droite']
).fillna(0).astype(int)

annot_db = hm_db.round(0).fillna(0).astype(int).astype(str) + '%'
annot_db = annot_db.where(hm_n_db >= 5, other='—')

sns.heatmap(
    hm_db, annot=annot_db, fmt='', cmap='RdYlGn_r',
    vmin=0, vmax=100, ax=ax,
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': '% négatif', 'shrink': 0.7},
    annot_kws={'size': 8}
)
ax.set_xticklabels(['Gauche', 'Centre', 'Droite'], rotation=0, fontsize=10)
ax.set_yticklabels(top20_depts, rotation=0, fontsize=8)
ax.set_xlabel('Bloc politique')
ax.set_title('(a) % négatif envers Mitterrand\ndépartement × bloc (top 25 depts critiques)',
             fontsize=11, fontweight='bold')

# 6b : Scatter % négatif gauche vs droite par département
ax2 = axes[1]

pct_g = dept_bloc[dept_bloc.index.get_level_values('bloc_politique') == 'gauche']['pct_neg']
pct_d = dept_bloc[dept_bloc.index.get_level_values('bloc_politique') == 'droite']['pct_neg']
n_g   = dept_bloc[dept_bloc.index.get_level_values('bloc_politique') == 'gauche']['total']
n_d   = dept_bloc[dept_bloc.index.get_level_values('bloc_politique') == 'droite']['total']

pct_g.index = pct_g.index.droplevel('bloc_politique')
pct_d.index = pct_d.index.droplevel('bloc_politique')
n_g.index   = n_g.index.droplevel('bloc_politique')
n_d.index   = n_d.index.droplevel('bloc_politique')

common = pct_g.index.intersection(pct_d.index)
# Filter: au moins 5 mentions dans chaque bloc
common = [d for d in common if n_g.get(d, 0) >= 5 and n_d.get(d, 0) >= 5]

scatter_x = pct_d.reindex(common).values
scatter_y = pct_g.reindex(common).values
sizes     = (n_g.reindex(common).values + n_d.reindex(common).values) / 3

sc = ax2.scatter(scatter_x, scatter_y, s=sizes, c=scatter_y - scatter_x,
                 cmap='RdBu_r', vmin=-100, vmax=100, alpha=0.7, edgecolors='#555', linewidth=0.3)

# Lignes de référence
ax2.axline((0, 0), slope=1, color='grey', linestyle='--', alpha=0.4, linewidth=1)
ax2.axhline(50, color='#e74c3c', linestyle=':', alpha=0.4, linewidth=1)
ax2.axvline(50, color='#2980b9', linestyle=':', alpha=0.4, linewidth=1)

# Annotations top depts
for dept in common[:12]:
    x_val = pct_d.get(dept, np.nan)
    y_val = pct_g.get(dept, np.nan)
    if not np.isnan(x_val) and not np.isnan(y_val):
        ax2.annotate(dept, (x_val, y_val), xytext=(4, 2),
                     textcoords='offset points', fontsize=7, color='#2c3e50')

plt.colorbar(sc, ax=ax2, label='Écart gauche − droite (pct négatif)', shrink=0.8)
ax2.set_xlabel('% négatif — bloc Droite')
ax2.set_ylabel('% négatif — bloc Gauche')
ax2.set_title('(b) % négatif Gauche vs Droite par département\nenvers Mitterrand (taille = n mentions)',
              fontsize=11, fontweight='bold')
ax2.xaxis.set_major_formatter(mticker.PercentFormatter())
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
ax2.grid(alpha=0.2)

fig.suptitle('Variation géographique du sentiment envers Mitterrand : rôle des blocs politiques',
             fontsize=13, fontweight='bold')
plt.tight_layout()

out6 = GRAPHS_DIR / 'fig6_geo_blocs.png'
fig.savefig(out6)
plt.show()
print(f'Sauvegardé : {out6}')

## Récapitulatif des fichiers produits

In [ ]:
import os
graphs = sorted(GRAPHS_DIR.glob('fig*.png'))
print('Figures produites :')
for g in graphs:
    size_kb = g.stat().st_size // 1024
    print(f'  {g.name:<40} {size_kb:>5} KB')
print(f'\nDossier : {GRAPHS_DIR}')